# BDC 2026 - Waste Image Classification (SigLIP ViT-B/16) — v2

Versi revisi dari notebook original. Perubahan:

1. **FIX bug StratifiedKFold** — fold yang dipakai untuk training sekarang ditentukan eksplisit lewat `CONFIG["fold"]`, bukan diam-diam jadi fold terakhir (bug lama: `train_df`/`valid_df` ditimpa terus di dalam loop `for fold, (...) in enumerate(skf.split(...))`).
2. **FIX urutan label** — target 0/1/2 dideteksi **otomatis** dari nama folder di `train/` (mis. `0_Recyclable`, `1_Electronic`, `2_Organic`), bukan lewat `LabelEncoder` alfabetis yang bisa salah urutan.
3. **Warmup + cosine LR schedule** (step per-batch, bukan per-epoch).
4. **Pilihan optimizer** lewat `CONFIG["optimizer"]`: `"adamw"` (default), `"adamw_llrd"` (layer-wise LR decay), atau `"lion"` (butuh `pip install lion-pytorch`).
5. **Test-Time Augmentation (TTA)** saat inference (`CONFIG["tta"]`).
6. **Evaluasi otomatis** terhadap `solution.csv` (F1, classification report, confusion matrix) untuk A/B testing tanpa perlu submit ke platform kompetisi.

**Requirements:**
```
pip install torch torchvision timm albumentations opencv-python pandas scikit-learn tqdm
# opsional, hanya kalau CONFIG["optimizer"] == "lion":
pip install lion-pytorch
```

In [1]:
import math
import os

import albumentations as A
import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

C:\Users\MyPC PRO\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

In [2]:
CONFIG = {
    "root": "BDC 2026",
    # class_order TIDAK di-hardcode -> dideteksi otomatis dari nama folder
    # di dalam train/ (mis. "0_Recyclable", "1_Electronic", "2_Organic").
    # Sort otomatis benar karena nama foldernya sudah diberi prefix angka.
    "img_size": 224,
    "batch_size": 16,
    "epochs": 20,
    "n_splits": 5,
    "fold": 0,                  # <-- integer 0..4, coba ganti untuk cek stabilitas split
    "seed": 42,
    "lr": 2e-5,
    "weight_decay": 1e-4,
    "warmup_ratio": 0.1,      # 10% step pertama dipakai untuk warmup
    "optimizer": "adamw",     # "adamw" | "adamw_llrd" | "lion"
    "llrd_decay": 0.9,        # dipakai kalau optimizer == "adamw_llrd"
    "tta": True,              # aktif/nonaktifkan TTA saat inference
    "model_name": "vit_base_patch16_siglip_224.v2_webli",
    "checkpoint": "best_model.pth",
    "submission_template": "BDC 2026/submission.csv",
    "submission_out": "submission_siglip_vit_v2.csv",
    "solution_csv": "solution.csv",  # opsional, untuk evaluasi lokal
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## Dataset

In [3]:
class WasteDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = cv2.imread(row["image"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform is not None:
            image = self.transform(image=image)["image"]
        return image, row["target"]


class TestDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_name = self.df.loc[idx, "image"]
        image_path = os.path.join(self.image_dir, image_name)
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image=image)["image"]
        return image, image_name

In [4]:
def get_transforms(img_size):
    train_transform = A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=20, p=0.5),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ])
    valid_transform = A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ])
    return train_transform, valid_transform

## 1) Fix: deteksi kelas otomatis + split fold eksplisit

`get_class_order` membaca nama folder di `train/` dan mengurutkannya. Karena foldernya sudah diberi prefix angka (`0_Recyclable`, `1_Electronic`, `2_Organic`), hasil sort selalu benar sesuai urutan target 0/1/2 — tidak perlu hardcode nama kelas lagi.

`get_fold_split` mengambil fold sesuai `CONFIG["fold"]` secara eksplisit (dulu bug-nya: variabel `train_df`/`valid_df` ditimpa terus tiap iterasi loop, jadi yang kepakai selalu fold terakhir).

In [5]:
def get_class_order(config):
    """Deteksi otomatis urutan kelas dari nama folder di train/.
    Folder harus diberi prefix angka (0_Recyclable, 1_Electronic, 2_Organic)
    supaya urutan hasil sort selalu benar."""
    train_dir = os.path.join(config["root"], "train")
    return sorted(os.listdir(train_dir))


def build_dataframe(train_dir, class_order):
    images, labels = [], []
    for c in class_order:
        folder = os.path.join(train_dir, c)
        for img in os.listdir(folder):
            images.append(os.path.join(folder, img))
            labels.append(c)
    df = pd.DataFrame({"image": images, "label": labels})

    label2id = {name: i for i, name in enumerate(class_order)}
    df["target"] = df["label"].map(label2id)
    return df


def get_fold_split(df, config):
    fold = config["fold"]
    if not float(fold).is_integer():
        raise ValueError(
            f'CONFIG["fold"] harus bilangan bulat (0..{config["n_splits"] - 1}), '
            f'sekarang bernilai {fold!r}. Cek lagi typo di CONFIG.'
        )
    fold = int(fold)
    if not (0 <= fold < config["n_splits"]):
        raise ValueError(f'CONFIG["fold"] harus di rentang 0..{config["n_splits"] - 1}, sekarang {fold}.')

    skf = StratifiedKFold(n_splits=config["n_splits"], shuffle=True, random_state=config["seed"])
    splits = list(skf.split(df, df["target"]))
    train_idx, valid_idx = splits[fold]
    train_df = df.iloc[train_idx].reset_index(drop=True)
    valid_df = df.iloc[valid_idx].reset_index(drop=True)
    return train_df, valid_df

## 2) Warmup + cosine LR schedule

In [6]:
def get_warmup_cosine_scheduler(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

## 3) Pilihan optimizer: AdamW / AdamW+LLRD / Lion

In [7]:
def build_llrd_param_groups(model, base_lr, decay, weight_decay):
    """Layer-wise LR decay: layer paling awal (patch embed) dapat lr paling
    kecil, layer paling akhir (head) dapat lr penuh (base_lr)."""
    num_layers = len(model.blocks)

    def get_layer_id(name):
        if name.startswith("patch_embed") or name in ("cls_token", "pos_embed"):
            return 0
        if name.startswith("blocks."):
            return int(name.split(".")[1]) + 1
        return num_layers + 1  # head / norm akhir

    groups = {}
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        layer_id = get_layer_id(name)
        lr = base_lr * (decay ** (num_layers + 1 - layer_id))
        if layer_id not in groups:
            groups[layer_id] = {"params": [], "lr": lr, "weight_decay": weight_decay}
        groups[layer_id]["params"].append(param)

    return list(groups.values())


def get_optimizer(model, config):
    name = config["optimizer"]
    if name == "lion":
        try:
            from lion_pytorch import Lion
        except ImportError as e:
            raise ImportError("Optimizer 'lion' butuh: pip install lion-pytorch") from e
        # Lion umumnya perlu lr lebih kecil & weight_decay lebih besar dibanding AdamW
        return Lion(model.parameters(), lr=config["lr"] * 0.3, weight_decay=config["weight_decay"] * 10)
    elif name == "adamw_llrd":
        param_groups = build_llrd_param_groups(model, config["lr"], config["llrd_decay"], config["weight_decay"])
        return torch.optim.AdamW(param_groups)
    elif name == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    else:
        raise ValueError(f"Optimizer tidak dikenal: {name}")

## Training & validation loop

In [8]:
def train_one_epoch(model, loader, criterion, optimizer, scheduler, device):
    model.train()
    running_loss = 0
    preds, labels = [], []

    progress = tqdm(loader, desc="Train")
    for images, target in progress:
        images = images.to(device)
        target = target.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        running_loss += loss.item()
        pred = torch.argmax(outputs, dim=1)
        preds.extend(pred.cpu().numpy())
        labels.extend(target.cpu().numpy())
        progress.set_description(f"Loss {loss.item():.4f}")

    epoch_loss = running_loss / len(loader)
    epoch_f1 = f1_score(labels, preds, average="macro")
    return epoch_loss, epoch_f1


@torch.no_grad()
def valid_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0
    preds, labels = [], []

    for images, target in loader:
        images = images.to(device)
        target = target.to(device)

        outputs = model(images)
        loss = criterion(outputs, target)

        running_loss += loss.item()
        pred = torch.argmax(outputs, dim=1)
        preds.extend(pred.cpu().numpy())
        labels.extend(target.cpu().numpy())

    epoch_loss = running_loss / len(loader)
    epoch_f1 = f1_score(labels, preds, average="macro")
    return epoch_loss, epoch_f1

In [9]:
def train(config):
    train_dir = os.path.join(config["root"], "train")
    class_order = get_class_order(config)
    print("Label mapping:", {name: i for i, name in enumerate(class_order)})

    df = build_dataframe(train_dir, class_order)
    train_df, valid_df = get_fold_split(df, config)
    print(f"Fold {config['fold']}: train={train_df.shape}, valid={valid_df.shape}")

    train_transform, valid_transform = get_transforms(config["img_size"])
    train_dataset = WasteDataset(train_df, train_transform)
    valid_dataset = WasteDataset(valid_df, valid_transform)

    train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True, num_workers=0)
    valid_loader = DataLoader(valid_dataset, batch_size=config["batch_size"], shuffle=False, num_workers=0)

    model = timm.create_model(config["model_name"], pretrained=True, num_classes=len(class_order))
    model = model.to(device)

    class_counts = df["target"].value_counts().sort_index().values.astype(float)
    weights = class_counts.sum() / (len(class_counts) * class_counts)
    weights = torch.tensor(weights, dtype=torch.float32).to(device)
    print("Class weights:", weights)
    criterion = nn.CrossEntropyLoss(weight=weights)

    optimizer = get_optimizer(model, config)

    total_steps = len(train_loader) * config["epochs"]
    warmup_steps = int(total_steps * config["warmup_ratio"])
    scheduler = get_warmup_cosine_scheduler(optimizer, warmup_steps, total_steps)
    print(f"Optimizer: {config['optimizer']} | Total steps: {total_steps} | Warmup steps: {warmup_steps}")

    best_f1 = 0
    for epoch in range(config["epochs"]):
        train_loss, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, device)
        valid_loss, valid_f1 = valid_one_epoch(model, valid_loader, criterion, device)

        print(f"\nEpoch {epoch + 1}/{config['epochs']}")
        print(f"Train Loss : {train_loss:.4f} | Train F1 : {train_f1:.4f}")
        print(f"Valid Loss : {valid_loss:.4f} | Valid F1 : {valid_f1:.4f}")

        if valid_f1 > best_f1:
            best_f1 = valid_f1
            torch.save(model.state_dict(), config["checkpoint"])
            print(f"Model terbaik disimpan (Valid F1: {best_f1:.4f})")

    print(f"\nTraining selesai. Best Valid F1: {best_f1:.4f}")

    # Classification report akhir pakai model terbaik (bukan model dari epoch terakhir)
    model.load_state_dict(torch.load(config["checkpoint"], map_location=device))
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for images, target in valid_loader:
            images = images.to(device)
            outputs = model(images)
            pred = torch.argmax(outputs, dim=1)
            preds.extend(pred.cpu().numpy())
            labels.extend(target.numpy())
    print("\n=== Classification Report (Validation, model terbaik) ===")
    print(classification_report(labels, preds, target_names=class_order))

## 4)+5) TTA & inference

In [10]:
def predict_tta(model, images):
    """images: tensor batch (B,C,H,W) yang sudah di-resize+normalize.
    Return softmax probs rata-rata dari beberapa augmentasi."""
    variants = [
        images,
        torch.flip(images, dims=[3]),            # horizontal flip
        torch.flip(images, dims=[2]),             # vertical flip
        torch.rot90(images, k=1, dims=[2, 3]),    # rotate 90 derajat
    ]
    probs_sum = None
    for v in variants:
        outputs = model(v)
        probs = torch.softmax(outputs, dim=1)
        probs_sum = probs if probs_sum is None else probs_sum + probs
    return probs_sum / len(variants)


def run_inference(config):
    class_order = get_class_order(config)
    test_dir = os.path.join(config["root"], "test")
    test_images = sorted(os.listdir(test_dir), key=lambda x: int("".join(filter(str.isdigit, x))))
    test_df = pd.DataFrame({"image": test_images})
    test_df["id"] = test_df["image"].apply(lambda x: int("".join(filter(str.isdigit, x))))

    _, valid_transform = get_transforms(config["img_size"])
    test_dataset = TestDataset(test_df, test_dir, valid_transform)
    test_loader = DataLoader(test_dataset, batch_size=config["batch_size"], shuffle=False, num_workers=0)

    model = timm.create_model(config["model_name"], pretrained=False, num_classes=len(class_order))
    model.load_state_dict(torch.load(config["checkpoint"], map_location=device))
    model.to(device)
    model.eval()

    predictions = []
    with torch.no_grad():
        for images, _ in tqdm(test_loader, desc="Inference"):
            images = images.to(device)
            if config["tta"]:
                probs = predict_tta(model, images)
            else:
                probs = torch.softmax(model(images), dim=1)
            preds = torch.argmax(probs, dim=1)
            predictions.extend(preds.cpu().numpy())

    pred_map = dict(zip(test_df["id"], predictions))
    submission = pd.read_csv(config["submission_template"])
    submission["predicted"] = submission["id"].map(pred_map)
    assert submission["predicted"].isna().sum() == 0, "Ada id yang tidak ter-mapping, cek ulang!"
    submission["predicted"] = submission["predicted"].astype(int)
    submission.to_csv(config["submission_out"], index=False)

    print(f"\nSubmission disimpan ke: {config['submission_out']}")
    print(submission["predicted"].value_counts())
    return submission

## 6) Evaluasi lokal pakai solution.csv

In [11]:
def evaluate_with_solution(config, submission):
    if not os.path.exists(config["solution_csv"]):
        print(f"\n({config['solution_csv']} tidak ditemukan, skip evaluasi lokal)")
        return

    class_order = get_class_order(config)
    gt = pd.read_csv(config["solution_csv"])
    gt["predicted"] = gt["predicted"].fillna(0).astype(int)  # NaN = kelas 0 (Recyclable)
    gt = gt.rename(columns={"predicted": "true_label"})

    eval_df = submission.merge(gt, on="id", how="left")
    y_true = eval_df["true_label"]
    y_pred = eval_df["predicted"]

    print("\n=== Evaluasi vs solution.csv ===")
    print("F1 Macro:", f1_score(y_true, y_pred, average="macro"))
    print()
    print(classification_report(y_true, y_pred, target_names=class_order))
    print()
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))

## Run

In [12]:
train(CONFIG)
submission = run_inference(CONFIG)
evaluate_with_solution(CONFIG, submission)

Label mapping: {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}
Fold 0: train=(19963, 3), valid=(4991, 3)


C:\Users\MyPC PRO\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Class weights: tensor([0.8713, 2.1058, 0.7260], device='cuda:0')
Optimizer: adamw | Total steps: 24960 | Warmup steps: 2496


Loss 0.0106: 100%|██████████| 1248/1248 [07:35<00:00,  2.74it/s]



Epoch 1/20
Train Loss : 0.2680 | Train F1 : 0.8756
Valid Loss : 0.0623 | Valid F1 : 0.9787
Model terbaik disimpan (Valid F1: 0.9787)


...


Epoch 19/20
Train Loss : 0.0015 | Train F1 : 0.9994
Valid Loss : 0.0518 | Valid F1 : 0.9897
Model terbaik disimpan (Valid F1: 0.9897)


Loss 0.0000: 100%|██████████| 1248/1248 [07:35<00:00,  2.74it/s]



Epoch 20/20
Train Loss : 0.0011 | Train F1 : 0.9996
Valid Loss : 0.0517 | Valid F1 : 0.9900
Model terbaik disimpan (Valid F1: 0.9900)

Training selesai. Best Valid F1: 0.9900

=== Classification Report (Validation, model terbaik) ===
              precision    recall  f1-score   support

0_Recyclable       0.98      0.99      0.99      1910
1_Electronic       0.99      0.99      0.99       790
   2_Organic       0.99      0.99      0.99      2291

    accuracy                           0.99      4991
   macro avg       0.99      0.99      0.99      4991
weighted avg       0.99      0.99      0.99      4991



C:\Users\MyPC PRO\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
Inference: 100%|██████████| 92/92 [00:38<00:00,  2.36it/s]


Submission disimpan ke: submission_siglip_vit_v2.csv
predicted
2    728
0    529
1    201
Name: count, dtype: int64

(solution.csv tidak ditemukan, skip evaluasi lokal)
